In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
import deepchem as dc

tasks, datasets, transformers = dc.molnet.load_tox21()
train_dataset, valid_dataset, test_dataset = datasets


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "C:\Users\elisa\.conda\envs\deepchem310\lib\runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\elisa\.conda\envs\deepchem310\lib\runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "C:\Users\elisa\.conda\envs\deepchem310\lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\elisa\.conda\envs\deepchem310\lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    a

In [3]:
print(train_dataset.X.shape)
print(train_dataset.y.shape)

print("Number of tasks:", len(tasks))
print("Train X shape:", train_dataset.X.shape)
print("Train y shape:", train_dataset.y.shape)
print("First 5 tasks:", tasks[:5])

(6258, 1024)
(6258, 12)
Number of tasks: 12
Train X shape: (6258, 1024)
Train y shape: (6258, 12)
First 5 tasks: ['NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 'NR-ER']


In [4]:
# importing the SMILES data from the dataset

smiles = train_dataset.ids
label = train_dataset.y

print(smiles[:5])
print(label[:5])

# convert them into dataframe
df = pd.DataFrame({
    "smiles": train_dataset.ids,
    "labels": train_dataset.y.tolist()
})

df.head()

['CC(O)(P(=O)(O)O)P(=O)(O)O' 'CC(C)(C)OOC(C)(C)CCC(C)(C)OOC(C)(C)C'
 'OC[C@H](O)[C@@H](O)[C@H](O)CO'
 'CCCCCCCC(=O)[O-].CCCCCCCC(=O)[O-].[Zn+2]' 'CC(C)COC(=O)C(C)C']
[[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]


,smiles,labels
0,CC(O)(P(=O)(O)O)P(=O)(O)O,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,CC(C)(C)OOC(C)(C)CCC(C)(C)OOC(C)(C)C,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
2,OC[C@H](O)[C@@H](O)[C@H](O)CO,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
3,CCCCCCCC(=O)[O-].CCCCCCCC(=O)[O-].[Zn+2],"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
4,CC(C)COC(=O)C(C)C,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


## Create a Baseline

#### Logistic Regression

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

task = 0   # choose one toxicity target

X_train = train_dataset.X
y_train = train_dataset.y[:, task]
w_train = train_dataset.w[:, task]

X_valid = valid_dataset.X
y_valid = valid_dataset.y[:, task]
w_valid = valid_dataset.w[:, task]

# remove missing labels
train_mask = w_train != 0
valid_mask = w_valid != 0

model = LogisticRegression(max_iter=1000)

model.fit(X_train[train_mask], y_train[train_mask])

pred = model.predict_proba(X_valid[valid_mask])[:,1]

auc = roc_auc_score(y_valid[valid_mask], pred)

print("Validation ROC-AUC:", auc)

Validation ROC-AUC: 0.7688793585679656


In [6]:
# using L2 regularization and class balancing

model_L2 = LogisticRegression(max_iter=1000, C=0.1, penalty="l2", class_weight="balanced")

model_L2.fit(X_train[train_mask], y_train[train_mask])

pred = model_L2.predict_proba(X_valid[valid_mask])[:,1]

auc = roc_auc_score(y_valid[valid_mask], pred)

print("Validation ROC-AUC after L2 regularization and class balancing:", auc)

Validation ROC-AUC after L2 regularization and class balancing: 0.7777363415998509


In [7]:
# looping on logistic regression

logreg_test_scores = []

for task_idx in range(len(tasks)):
    X_train = train_dataset.X
    y_train = train_dataset.y[:, task_idx]
    w_train = train_dataset.w[:, task_idx]

    X_valid = valid_dataset.X
    y_valid = valid_dataset.y[:, task_idx]
    w_valid = valid_dataset.w[:, task_idx]

    X_test = test_dataset.X
    y_test = test_dataset.y[:, task_idx]
    w_test = test_dataset.w[:, task_idx]

    train_mask = w_train != 0
    valid_mask = w_valid != 0
    test_mask = w_test != 0

    # combine train + valid for final training
    X_combined = np.concatenate([X_train[train_mask], X_valid[valid_mask]], axis=0)
    y_combined = np.concatenate([y_train[train_mask], y_valid[valid_mask]], axis=0)

    if len(np.unique(y_combined)) < 2 or len(np.unique(y_test[test_mask])) < 2:
        logreg_test_scores.append(np.nan)
        continue

    model = LogisticRegression(
        penalty="l2",
        class_weight="balanced",
        max_iter=1000
    )

    model.fit(X_combined, y_combined)
    pred = model.predict_proba(X_test[test_mask])[:, 1]
    auc = roc_auc_score(y_test[test_mask], pred)
    logreg_test_scores.append(auc)

print("LogReg test ROC-AUC:", logreg_test_scores)
print("LogReg mean test ROC-AUC:", np.nanmean(logreg_test_scores))

LogReg test ROC-AUC: [0.6466946597760551, 0.6919530230535016, 0.7329366043235365, 0.6871535848381907, 0.6560064935064934, 0.7267932489451477, 0.5869636692421503, 0.644242891161227, 0.709299568454498, 0.689483282674772, 0.7038620283018867, 0.6655963759458382]
LogReg mean test ROC-AUC: 0.6784154525186081


#### Random Forest

In [7]:
from sklearn.ensemble import RandomForestClassifier

model_rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    n_jobs=-1,
    class_weight="balanced",
    random_state=0
)

model_rf.fit(X_train[train_mask], y_train[train_mask])

pred = model_rf.predict_proba(X_valid[valid_mask])[:,1]

auc = roc_auc_score(y_valid[valid_mask], pred)

print("Validation ROC-AUC RF model:", auc)

Validation ROC-AUC RF model: 0.7720725340294611


In [8]:
# multi-task QSAR modeling

rf_scores = []

for task in range(len(tasks)):

    X_train = train_dataset.X
    y_train = train_dataset.y[:, task]
    w_train = train_dataset.w[:, task]

    X_valid = valid_dataset.X
    y_valid = valid_dataset.y[:, task]
    w_valid = valid_dataset.w[:, task]

    train_mask = w_train != 0
    valid_mask = w_valid != 0

    # skip tasks without both classes
    if len(np.unique(y_train[train_mask])) < 2:
        rf_scores.append(np.nan)
        continue

    model = RandomForestClassifier(
        n_estimators=500,
        max_depth=None,
        class_weight="balanced",
        n_jobs=-1,
        random_state=0
    )

    model.fit(X_train[train_mask], y_train[train_mask])

    pred = model.predict_proba(X_valid[valid_mask])[:,1]

    auc = roc_auc_score(y_valid[valid_mask], pred)

    rf_scores.append(auc)

print("Per-task ROC-AUC:", rf_scores)
print("Mean ROC-AUC:", np.nanmean(rf_scores))

Per-task ROC-AUC: [0.7720725340294611, 0.8419269102990032, 0.7876263935701322, 0.7229885057471265, 0.6382045929018789, 0.7945696443117025, 0.7694738051470588, 0.7505912292427126, 0.6890529184646832, 0.7159305317324186, 0.8052284022057321, 0.7759045956951717]
Mean ROC-AUC: 0.7552975052789234


#### DeepChem neural baseline
##### Multitask Classifier

In [3]:
n_tasks = len(tasks)
n_features = train_dataset.X.shape[1]

model = dc.models.MultitaskClassifier(
    n_tasks=n_tasks,
    n_features=n_features,
    layer_sizes=[1000],
    dropouts=0.25,
    learning_rate=0.001
)

model.fit(train_dataset, nb_epoch=10)

metric = dc.metrics.Metric(dc.metrics.roc_auc_score)

valid_scores = model.evaluate(valid_dataset, [metric], transformers)
test_scores = model.evaluate(test_dataset, [metric], transformers)

print("NN validation:", valid_scores)
print("NN test:", test_scores)

NN validation: {'roc_auc_score': 0.7034002443219859}
NN test: {'roc_auc_score': 0.6762468284358097}


##### Graph Neural Network

In [8]:
import deepchem as dc
from deepchem.models import GCNModel

featurizer = dc.feat.MolGraphConvFeaturizer()

tasks, datasets, transformers = dc.molnet.load_tox21(
    featurizer=featurizer
)

train_dataset, valid_dataset, test_dataset = datasets

model = GCNModel(
    n_tasks=len(tasks),
    mode="classification",
    batch_size=64
)

model.fit(train_dataset, nb_epoch=10)

0.7386758327484131

In [ ]:
metric = dc.metrics.Metric(dc.metrics.roc_auc_score)

model.evaluate(valid_dataset, [metric], transformers)

##### Message Passing Neural Network

In [ ]:
model = dc.models.MPNNModel(
    n_tasks=len(tasks),
    mode="classification"
)